In [0]:
-- ### user defined functions. In short UDF ###.
dbo: database owner

-- ### In SQL Server there are 3 types of User Defined functions
1. Scalar functions
2. Inline table-valued functions
3. Multistatement table-valued functions

-- 1. What is Scaler Function?
Scalar functions may or may not have parameters, but always return a single (scalar) value. The returned value can be of any data type, except text, ntext, image, cursor, and timestamp.

-- 2. How to create Function in SQl? CREATE FUNCTION statement?
CREATE FUNCTION Function_Name(@Parameter1 DataType, @Parameter2 DataType,..@Parametern Datatype)
RETURNS Return_Datatype
AS
BEGIN
    -- Function Body
    Return Return_Datatype
END

2. Real Example of Function => Find AGE from Date of birth?
CREATE FUNCTION Age(@DOB Date)  
RETURNS INT  
AS  
BEGIN  
 -- Function Body
 DECLARE @Age INT  
 SET @Age = DATEDIFF(YEAR, @DOB, GETDATE()) - 
  CASE 
    WHEN (MONTH(@DOB) > MONTH(GETDATE())) OR 
    (MONTH(@DOB) = MONTH(GETDATE()) AND DAY(@DOB) > DAY(GETDATE())) 
    THEN 1 
    ELSE 0 
  END  
 RETURN @Age  
END

-- When calling a scalar user-defined function, you must supply a two-part name, OwnerName.FunctionName. dbo stands for database owner.
Select dbo.Age(dbo.Age('10/08/1982')
-- You can also invoke it using the complete 3 part name, DatabaseName.OwnerName.FunctionName.
Select SampleDB.dbo.Age('10/08/1982')

-- DATABRICKS
Key Differences from SQL Server to Databricks:
Function Definition: CREATE FUNCTION syntax is similar but Databricks uses RETURN instead of BEGIN...END for simple functions
Date Functions:
GETDATE() → current_date() or current_timestamp()
DATEDIFF() → datediff()
MONTH() → month()
DAY() → day()
Calling Function: No need for dbo. prefix in Databricks

-- Method 1: Databricks SQL UDF (Scalar Function)
CREATE OR REPLACE FUNCTION Age(DOB DATE)
RETURNS INT
RETURN year(current_date()) - year(DOB) - 
       CASE 
         WHEN month(current_date()) < month(DOB) OR 
              (month(current_date()) = month(DOB) AND day(current_date()) < day(DOB)) 
         THEN 1 
         ELSE 0 
       END;

-- Test the function
SELECT Age(date('2001-04-04')) as Age;

-- INPUT
Id | Name | DateOfBirth
1  | Sam  | 1980-12-30 00:00:00.000
2  | Pam  | 1982-09-01 12:02:36.260
3  | John | 1985-08-22 12:03:30.370
4  | Sara | 1979-11-29 12:59:30.670
--OUTPUT1
Select Name, DateOfBirth, dbo.Age(DateOfBirth) as Age from tblEmployees;

Name | DateOfBirth                | Age
Sam  | 1980-12-30 00:00:00.000    | 31
Pam  | 1982-09-01 12:02:36.260    | 30
John | 1985-08-22 12:03:30.370    | 27
Sara | 1979-11-29 12:59:30.670    | 32
--OUTPUT2
Select Name, DateOfBirth, dbo.Age(DateOfBirth) as Age 
from tblEmployees
Where dbo.Age(DateOfBirth) > 30

Name | DateOfBirth                | Age
Sam  | 1980-12-30 00:00:00.000    | 31
Sara | 1979-11-29 12:59:30.670    | 32
 